# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [14]:
aviation_data=pd.read_csv("data/AviationData.csv", encoding= "latin1")

aviation_data.shape
aviation_data.columns
aviation_data.head()
aviation_data.info()
aviation_data.describe()
aviation_data.isna().sum()


C:\Users\user\AppData\Local\Temp\ipykernel_17584\4286186228.py:1: DtypeWarning: Columns (0: Latitude, 1: Longitude, 2: Broad.phase.of.flight) have mixed types. Specify dtype option on import or set low_memory=False.
  aviation_data=pd.read_csv("data/AviationData.csv", encoding= "latin1")


<class 'pandas.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  str    
 1   Investigation.Type      88889 non-null  str    
 2   Accident.Number         88889 non-null  str    
 3   Event.Date              88889 non-null  str    
 4   Location                88837 non-null  str    
 5   Country                 88663 non-null  str    
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  str    
 9   Airport.Name            52704 non-null  str    
 10  Injury.Severity         87889 non-null  str    
 11  Aircraft.damage         85695 non-null  str    
 12  Aircraft.Category       32287 non-null  str    
 13  Registration.Number     87507 non-null  str    
 14  Make                    88826 non-null  str    
 

Event.Id                      0
Investigation.Type            0
Accident.Number               0
Event.Date                    0
Location                     52
Country                     226
Latitude                  54507
Longitude                 54516
Airport.Code              38757
Airport.Name              36185
Injury.Severity            1000
Aircraft.damage            3194
Aircraft.Category         56602
Registration.Number        1382
Make                         63
Model                        92
Amateur.Built               102
Number.of.Engines          6084
Engine.Type                7096
FAR.Description           56866
Schedule                  76307
Purpose.of.flight          6192
Air.carrier               72241
Total.Fatal.Injuries      11401
Total.Serious.Injuries    12510
Total.Minor.Injuries      11933
Total.Uninjured            5912
Weather.Condition          4492
Broad.phase.of.flight     27165
Report.Status              6384
Publication.Date          13771
dtype: i

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [15]:
aviation_data.info()
aviation_data.head()
# Convert Event.Date to datetime
aviation_data["Event.Date"] = pd.to_datetime(aviation_data["Event.Date"], errors="coerce")
# Fill missing injury counts with 0
injury_cols = ["Total.Fatal.Injuries","Total.Serious.Injuries",
               "Total.Minor.Injuries","Total.Uninjured"]
aviation_data[injury_cols] = aviation_data[injury_cols].fillna(0)

# Drop rows missing essential identifiers
aviation_data = aviation_data.dropna(subset=["Make","Model","Aircraft.damage","Event.Date"])
# Keep only accidents from 1983–2023
aviation_data = aviation_data[
    (aviation_data["Event.Date"].dt.year >= 1983) &
    (aviation_data["Event.Date"].dt.year <= 2023)
]

# Exclude amateur-built aircraft
aviation_data = aviation_data[
    aviation_data["Amateur.Built"].str.upper() != "YES"
]


<class 'pandas.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  str    
 1   Investigation.Type      88889 non-null  str    
 2   Accident.Number         88889 non-null  str    
 3   Event.Date              88889 non-null  str    
 4   Location                88837 non-null  str    
 5   Country                 88663 non-null  str    
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  str    
 9   Airport.Name            52704 non-null  str    
 10  Injury.Severity         87889 non-null  str    
 11  Aircraft.damage         85695 non-null  str    
 12  Aircraft.Category       32287 non-null  str    
 13  Registration.Number     87507 non-null  str    
 14  Make                    88826 non-null  str    
 

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [16]:
# Fill missing injury counts with 0. if injury counts are missing, we treat them as 0.
injury_cols = ["Total.Fatal.Injuries","Total.Serious.Injuries",
               "Total.Minor.Injuries","Total.Uninjured"]
aviation_data[injury_cols] = aviation_data[injury_cols].fillna(0)
# calculate the total number of people on board by summing all injury categories plus uninjured.
aviation_data["Total_Occupants"] = (
    aviation_data["Total.Fatal.Injuries"] +
    aviation_data["Total.Serious.Injuries"] +
    aviation_data["Total.Minor.Injuries"] +
    aviation_data["Total.Uninjured"]
)
#Assumption: dataset records all passengers as either injured (fatal, serious, minor) or uninjured.
# measure the likelihood of severe injury as a fraction of total occupants. Formula: (Fatal + Serious Injuries) ÷ Total
aviation_data["FatalSerious_Fraction"] = (
    (aviation_data["Total.Fatal.Injuries"] + aviation_data["Total.Serious.Injuries"]) /
    aviation_data["Total_Occupants"].replace(0, np.nan)
)

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [17]:
# Standardize Aircraft.damage labels
aviation_data["Aircraft.damage"] = aviation_data["Aircraft.damage"].str.strip().str.title()

# Drop rows where Aircraft.damage is missing 
# Rows missing damage information are dropped because this field is central to the client’s analysis.
aviation_data = aviation_data.dropna(subset=["Aircraft.damage"])
# Create binary indicator for destruction
#"Destroyed" → 1, all other categories → 0.
aviation_data["Destroyed"] = aviation_data["Aircraft.damage"].eq("Destroyed").astype(int)
#only "Destroyed" counts as total loss; "Substantial" or "Minor" do not.


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [18]:
# Standardize capitalization and strip whitespace
aviation_data["Make"] = aviation_data["Make"].str.strip().str.title()

# Count how many accident records exist for each manufacturer (Make).
# This helps us decide which manufacturers have enough data to analyze reliably.
make_counts = aviation_data["Make"].value_counts()

# Keep only manufacturers with at least 50 accident records.
# Rationale: a minimum threshold ensures statistical robustness,
# so to avoid making conclusions based on very few data points.
aviation_data = aviation_data[aviation_data["Make"].isin(make_counts[make_counts >= 50].index)]

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [19]:
# Drop rows with missing Model
aviation_data = aviation_data.dropna(subset=["Model"])

# Standardize capitalization and strip whitespace
aviation_data["Model"] = aviation_data["Model"].str.strip().str.upper()

# Create a unique identifier combining Make and Model
aviation_data["Make_Model"] = aviation_data["Make"] + "_" + aviation_data["Model"]

# Inspect counts for each Make_Model combination
model_counts = aviation_data["Make_Model"].value_counts()

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [20]:
# Strip extra spaces and standardize capitalization (e.g., "turbojet ", "TURBOJET" → "Turbojet")
aviation_data["Engine.Type"] = aviation_data["Engine.Type"].str.strip().str.title()

# Clean Weather.Condition column:
aviation_data["Weather.Condition"] = aviation_data["Weather.Condition"].str.strip().str.title()

# Clean Purpose.of.flight column:
aviation_data["Purpose.of.flight"] = aviation_data["Purpose.of.flight"].str.strip().str.title()

# Clean Broad.phase.of.flight column:
aviation_data["Broad.phase.of.flight"] = aviation_data["Broad.phase.of.flight"].str.strip().str.title()

# Ensure Number.of.Engines is numeric:
aviation_data["Number.of.Engines"] = pd.to_numeric(aviation_data["Number.of.Engines"], errors="coerce")


### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [ ]:
# Check how many missing values each column has
missing_counts = aviation_data.isnull().sum()

# Calculate percentage of missing values per column
missing_percentage = (missing_counts / len(aviation_data)) * 100

# Display summary of missing values
print(missing_percentage.sort_values(ascending=False))


# Drop columns with more than 40% missing values ,columns with too many NaNs are not reliable for analysis
cols_to_drop = missing_percentage[missing_percentage > 40].index
aviation_data = aviation_data.drop(columns=cols_to_drop)


Schedule                  86.411325
Air.carrier               82.464462
FAR.Description           70.310607
Aircraft.Category         70.068268
Longitude                 62.864616
Latitude                  62.855750
Airport.Code              42.681089
Airport.Name              39.982859
Broad.phase.of.flight     27.925052
Publication.Date          17.281378
Engine.Type                6.587463
Report.Status              6.082099
Purpose.of.flight          5.757012
Number.of.Engines          5.677217
Weather.Condition          4.011880
Registration.Number        1.485061
FatalSerious_Fraction      0.877737
Injury.Severity            0.622100
Country                    0.268936
Amateur.Built              0.106392
Location                   0.063540
Event.Id                   0.000000
Investigation.Type         0.000000
Accident.Number            0.000000
Make                       0.000000
Aircraft.damage            0.000000
Event.Date                 0.000000
Model                      0

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [22]:
aviation_data.to_csv("data/AviationData_Cleaned.csv", index=False)
# index=False' prevents pandas from writing row numbers into the file. This makes the CSV cleaner and easier to reload later.